# Notebook 1B — Data Preprocessing (Section 1.1)

Covers: AbRank HCDR3 extraction, ANARCI alignment, contrastive/fine-tune splits,
UniProt antigen fetching, ESM2 antigen embeddings.

**Prerequisite:** Run Notebook 1A first so `df_section1.csv` exists on Drive.

**Outputs saved to Drive:**
- `abrank_stage1.csv` — 70k valid AbRank pairs for InfoNCE pretraining
- `abrank_stage2.csv` — multi-variant pairs for MSE fine-tuning
- `anarci_heavy_aho.json` — ANARCI AHo alignment cache
- `clinical_mabs_with_antigens.csv` — clinical mAbs with antigen sequences
- ESM2 antigen embedding caches (`.npy` files)


In [ ]:
# Install required packages.
# Run this cell once. If Colab prompts to restart the runtime, do so
# and then skip back to this cell (it will already be installed).
!pip install -q ablang2        # AbLang2 antibody language model
!pip install -q torchcfm       # Conditional flow matching training loop
!pip install -q torchdiffeq    # ODE integrator for Phase 1 sampling
!pip install -q scikit-learn scipy matplotlib seaborn joblib
!pip install -q fair-esm       # ESM2 antigen embeddings
!pip install -q anarci         # IMGT CDR3 extraction for AbRank / SAbDab sequences
!pip install -q requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.3 MB/s eta 0:00:00


In [ ]:
!apt-get install -y hmmer


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libdivsufsort3
Suggested packages:
  hmmer-doc
The following NEW packages will be installed:
  hmmer libdivsufsort3
0 upgraded, 2 newly installed, 0 to remove and 42 not upgraded.
Need to get 1,198 kB of archives.
After this operation, 7,621 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libdivsufsort3 amd64 2.0.1-5 [42.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 hmmer amd64 3.3.2+dfsg-1 [1,155 kB]
Fetched 1,198 kB in 1s (1,674 kB/s)
Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_am

In [ ]:
import os, re, warnings, json, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import spearmanr, pearsonr, ttest_rel, ks_2samp
from scipy.spatial.distance import cdist as scipy_cdist
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
import joblib
import requests
from difflib import get_close_matches

import esm
from anarci import anarci as anarci_number

from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
from torchdiffeq import odeint

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Directory and file paths ──────────────────────────────────────────────────
PROJECT_DIR = '/content/drive/MyDrive/2026 Spring/BMI 702/project'
EVAL_DIR    = f'{PROJECT_DIR}/eval_inputs'
DATA_PATH   = f'{PROJECT_DIR}/GDPa1_v1.2_20250814.csv'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,    exist_ok=True)

CACHE = lambda name: f'{PROJECT_DIR}/{name}'

# ── Dataset column names ──────────────────────────────────────────────────────
COL_VH    = 'vh_protein_sequence'
COL_VL    = 'vl_protein_sequence'
COL_HIC   = 'HIC'
COL_SINS  = 'AC-SINS_pH7.4'
COL_FOLD  = 'hierarchical_cluster_fold'
COL_AHO_H = 'heavy_aligned_aho'
COL_AHO_L = 'light_aligned_aho'
COL_NAME  = 'antibody_name'

# ── Pipeline hyperparameters ──────────────────────────────────────────────────
TEST_FOLD       = 1
N_FLOW_SAMPLES  = 50
N_ODE_STEPS     = 100
M_SEQUENCES     = 20
TEMPERATURE     = 1.0
TOP_K_SELECT    = 5
FLOW_EPOCHS     = 1000
CURRENT_SIGMA   = 0.5
PLS_COMPONENTS  = 10
AA_VOCAB        = list('ACDEFGHIKLMNPQRSTVWY')
AA_TO_IDX       = {aa: i for i, aa in enumerate(AA_VOCAB)}
ABLANG_MASK     = '*'

# ── Binding scorer hyperparameters ────────────────────────────────────────────
SCORER_HIDDEN      = 128    # shared projection dim (CDR3 and antigen)
SCORER_HEADS       = 4      # multi-head cross-attention heads
SCORER_PRETRAIN_EP = 100    # InfoNCE pretraining epochs
SCORER_FINETUNE_EP = 50     # affinity regression fine-tuning epochs
SCORER_BATCH       = 64     # contrastive pretraining batch size
ANTIGEN_MAX_LEN    = 512    # truncate antigen sequences for ESM2
GUIDANCE_LAMBDA    = 0.5    # binding gradient scale at ODE sampling time
                             # 0 = no guidance (ablation), >1 = stronger push
ESM2_DIM           = 1280   # ESM2-650M hidden dimension

Mounted at /content/drive


In [ ]:
MEAN_CACHE       = f'{EVAL_DIR}/antigen_esm2_embs.npy'
RESID_CACHE      = f'{EVAL_DIR}/antigen_esm2_resid.npy'
MASK_CACHE       = f'{EVAL_DIR}/antigen_valid_mask.npy'
RESID_PAD_CACHE  = f'{EVAL_DIR}/antigen_resid_pad_mask.npy'
SAB_MEAN_CACHE   = f'{EVAL_DIR}/sabdab_ag_embs.npy'
SAB_CDR3_CACHE   = f'{EVAL_DIR}/sabdab_cdr3_embs.npy'

In [ ]:
# ── Reload df from Notebook 1A checkpoint ───────────────────────────────────
# This avoids re-running the full Section 1 pipeline.
# Make sure Notebook 1A has been run and df_section1.csv exists on Drive.
df = pd.read_csv(f'{PROJECT_DIR}/df_section1.csv')

# Restore list columns that were serialised as strings by to_csv
import ast
for col in ['h_cdr3_idx', 'h_fw_idx']:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Rebuild train/test index masks (same logic as Section 1c)
train_mask = df[COL_FOLD] != TEST_FOLD
test_mask  = df[COL_FOLD] == TEST_FOLD
train_idx  = df.index[train_mask].values
test_idx   = df.index[test_mask].values

print(f'Reloaded df: {len(df)} rows')
print(f'Train: {len(train_idx)}  |  Test: {len(test_idx)}')
print(f'Columns: {df.columns.tolist()}')


Reloaded df: 246 rows
Train: 196  |  Test: 50
Columns: ['antibody_id', 'antibody_name', 'Titer', 'Purity', 'SEC %Monomer', 'SMAC', 'HIC', 'HAC', 'PR_CHO', 'PR_Ova', 'AC-SINS_pH6.0', 'AC-SINS_pH7.4', 'Tonset', 'Tm1', 'Tm2', 'hc_subtype', 'lc_subtype', 'highest_clinical_trial_asof_feb2025', 'est_status_asof_feb2025', 'vh_protein_sequence', 'hc_protein_sequence', 'hc_dna_sequence', 'vl_protein_sequence', 'lc_protein_sequence', 'lc_dna_sequence', 'hierarchical_cluster_fold', 'random_fold', 'hierarchical_cluster_IgG_isotype_stratified_fold', 'light_aligned_aho', 'heavy_aligned_aho', 'hcdr3_sequence', 'h_cdr3_idx', 'h_fw_idx', 'hcdr3_len', 'liability_count', 'liability_detail']


---
## Section 1.1 — AbRank Affinity Dataset & Clinical mAb Kd Lookup

Uses the **AbRank dataset** (`df_match`) as the source of Ab–Ag pairs with
experimental affinity labels. This replaces the SAbDab download entirely.

Two outputs:
1. `sabdab_valid` — Ab–Ag pairs for binding scorer training (Section 5.1–5.4)
2. `clinical_kd_df` — Kd values extracted for clinical mAbs in our dataset
   (via sequence matching against AbRank)

First, test anarci and verify that we get the same output of aho format as ginkgo

In [ ]:
from anarci import run_anarci
#from tqdm.notebook import tqdm

# test anarchi output matches with Ginkgo data
heavy_sequences = []
test = df.loc[:2, ['vh_protein_sequence', 'hc_protein_sequence', 'vl_protein_sequence', 'lc_protein_sequence', 'light_aligned_aho', 'heavy_aligned_aho']]

for i, seq in enumerate(test['hc_protein_sequence']):
    results = run_anarci([(i, seq)], scheme="aho")
    try:
        residues = [aa for (pos, ins), aa in results[1][0][0][0] ]
        heavy_sequences.append(''.join(residues))
    except (TypeError, IndexError):
        # ANARCI returns None for sequences it can't number
        heavy_sequences.append(None)


test_anarci = pd.DataFrame(heavy_sequences, columns = ['aho_heavy'])
test['heavy_aligned_aho'].equals(test_anarci['aho_heavy'])

True

In [ ]:
# ── Load AbRank dataset ───────────────────────────────────────────────────────
ABRANK_PATH = f'{PROJECT_DIR}/antigen/AbRank_dataset.csv'
df_match = pd.read_csv(ABRANK_PATH, sep='\t')
df_match = df_match.dropna(subset=[
    'Ab_heavy_chain_seq', 'Ab_light_chain_seq',
    'Ag_seq'
]).reset_index(drop=True)

print(f'AbRank entries loaded: {len(df_match)}')
print(f'Columns: {list(df_match.columns)}')
print(f'Entries with Affinity_Kd [nM]: {df_match["Affinity_Kd [nM]"].notna().sum()}')
print(f'Entries with log_Aff:          {df_match["log_Aff"].notna().sum() if "log_Aff" in df_match.columns else "N/A"}') #Kd ratio between mutant and wildtype

AbRank entries loaded: 342356
Columns: ['Ab_name', 'Ag_name', 'Ag_name_details', 'IC50 [ug/mL]', 'Affinity_Kd [nM]', 'Ag_epitope_restrictions', 'Ab_heavy_chain_seq', 'Ab_light_chain_seq', 'Ag_seq', 'Ab_structure_method', 'Ag_structure_method', 'bound_AbAg_structure_method', 'Ab_PDB_ID', 'Ag_PDB_ID', 'bound_AbAg_PDB_ID', 'Ab_Lev3_cluster', 'Source', 'escape', 'log(Kd_ratio)', 'Ab10_cluster', 'Ab25_cluster', 'Ab50_cluster', 'Ag_Lev3_cluster', 'Ag10_cluster', 'Ag25_cluster', 'Ag50_cluster', 'log_Aff', 'Aff_op']
Entries with Affinity_Kd [nM]: 94686
Entries with log_Aff:          169238


verify matching heavy chain from ginkgo data and abrank

### 1.1a — Extract HCDR3 from AbRank heavy chains (IMGT/ANARCI)

In [ ]:
import re

def clean_seq(seq):
    if not isinstance(seq, str):
        return None
    # Keep only standard amino acid characters
    cleaned = re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', '', seq.upper().strip())
    return cleaned if len(cleaned) >= 30 else None

# Check what illegal characters exist first
all_chars = set(''.join(df_match['Ab_heavy_chain_seq'].dropna().tolist()))

valid_aa   = set('ACDEFGHIKLMNPQRSTVWY')
illegal    = all_chars - valid_aa
print(f'Illegal characters found: {illegal}')

df_match['hc_clean'] = df_match['Ab_heavy_chain_seq'].apply(clean_seq)


Illegal characters found: {'5', '2', '+', ' ', 'l', '3'}


In [ ]:
# get unique heavy chain seq
unique_hc = df_match['hc_clean'].dropna().unique()
print(f'Unique heavy chain sequences to align: {len(unique_hc)}')  # ~31k not 342k

Unique heavy chain sequences to align: 31617


In [ ]:
import os, json
from tqdm.notebook import tqdm

ANARCI_CACHE = f'{PROJECT_DIR}/anarci_heavy_aho.json'

def batch_anarci(sequences, batch_size=256):
    """Align unique sequences in batches. Returns dict {seq: aho_str}."""
    seq_list = list(sequences)
    results_dict = {}

    for start in tqdm(range(0, len(seq_list), batch_size), desc='ANARCI heavy'):
        batch = [(j, seq) for j, seq in enumerate(seq_list[start:start+batch_size])]
        try:
            output = run_anarci(batch, scheme='aho', output=False, ncpu=1)
            numbered = output[1]  # index 1 = per-sequence numbering
            for (_, seq), result in zip(batch, numbered):
                if result is None or result[0] is None:
                    results_dict[seq] = None
                    continue
                residues = [aa for (pos, ins), aa in result[0][0]]
                results_dict[seq] = ''.join(residues)
        except Exception as e:
            print(f'Batch {start} failed: {e}')
            for _, seq in batch:
                results_dict[seq] = None
    return results_dict

# Run or load from cache
if os.path.exists(ANARCI_CACHE):
    print('Loading from cache...')
    with open(ANARCI_CACHE) as f:
        vh_aho_dict = json.load(f)
    print(f'Loaded {len(vh_aho_dict)} entries')
else:
    print(f'Aligning {len(unique_hc)} unique hc sequences...')
    vh_aho_dict = batch_anarci(unique_hc)
    with open(ANARCI_CACHE, 'w') as f:
        json.dump(vh_aho_dict, f)
    print(f'Saved to cache.')

# Map back to full dataframe
df_match['heavy_aligned_aho'] = df_match['hc_clean'].map(vh_aho_dict)
print(f"Aligned: {df_match['heavy_aligned_aho'].notna().sum()} / {len(df_match)}")

Loading from cache...
Loaded 31617 entries
Aligned: 342356 / 342356


In [ ]:
AHO_CDR3_START = 106
AHO_CDR3_END   = 138

def extract_hcdr3_from_aho(vh_seq, aho_str):
    if not isinstance(aho_str, str) or not isinstance(vh_seq, str):
        return None
    cdr3_chars = []
    raw_pos = 0
    for aho_pos, char in enumerate(aho_str):
        if char == '-':
            continue
        if AHO_CDR3_START <= aho_pos < AHO_CDR3_END:
            if raw_pos < len(vh_seq):
                cdr3_chars.append(vh_seq[raw_pos])
        raw_pos += 1
    return ''.join(cdr3_chars) if cdr3_chars else None

df_match['hcdr3'] = df_match.apply(
    lambda r: extract_hcdr3_from_aho(r['hc_clean'], r['heavy_aligned_aho']), axis=1)

# ── Diagnostics ───────────────────────────────────────────────────────────────
print(f"Valid HCDR3: {df_match['hcdr3'].notna().sum()} / {len(df_match)}")
print(f"Empty HCDR3: {(df_match['hcdr3'] == '').sum()}")
df_match_check = df_match.copy()
# Length distribution — flag anything outside normal range (6-24 aa)
df_match_check['hcdr3_len'] = df_match['hcdr3'].apply(
    lambda s: len(s) if isinstance(s, str) else 0)

print(f"\nHCDR3 length distribution:")
print(df_match_check['hcdr3_len'].describe().round(1))

Valid HCDR3: 341764 / 342356
Empty HCDR3: 0

HCDR3 length distribution:
count    342356.0
mean         16.3
std           5.2
min           0.0
25%          12.0
50%          16.0
75%          20.0
max          32.0
Name: hcdr3_len, dtype: float64


In [ ]:
suspect = df_match_check[(df_match_check['hcdr3_len'] < 6) | (df_match_check['hcdr3_len'] > 24)]
suspect_unique = suspect.drop_duplicates(subset='hcdr3')
print(f"Suspect HCDR3 lengths (<6 or >24 aa): {len(suspect_unique)} unique / {len(suspect)} total rows (with hcdr length out of range)")
print(suspect_unique[['hc_clean', 'heavy_aligned_aho', 'hcdr3', 'hcdr3_len']].head(5))

Suspect HCDR3 lengths (<6 or >24 aa): 136 unique / 25116 total rows (with hcdr length out of range)
                                               hc_clean  \
12    QVHLQESGPGLVKPSETLSLTCNVSGTLVRDNYWSWIRQPLGKQPE...   
43    EVQLVQSGAEVKKRGSSVKVSCKSSGGTFSNYAINWVRQAPGQGLE...   
58    GVQLVESGGGLVQPGRSLRLSCAASGFTFSNYAMYWVRQAPGKGLE...   
2150  QVQLVQSGAEVKKPGSSVKVSCKASGGTFSSYAISWVRQAPGQGLE...   
2158  EVQLVESGGGLFQPGGSLRLSCAASGFSVRNNYVSWVRQAPGKGLE...   

                                      heavy_aligned_aho  \
12    QVHLQES-GPGLVKPSETLSLTCNVSG-TLVRD-----NYWSWIRQ...   
43    EVQLVQS-GAEVKKRGSSVKVSCKSSG-GTFSN-----YAINWVRQ...   
58    GVQLVES-GGGLVQPGRSLRLSCAASG-FTFSN-----YAMYWVRQ...   
2150  QVQLVQS-GAEVKKPGSSVKVSCKASG-GTFSS-----YAISWVRQ...   
2158  EVQLVES-GGGLFQPGGSLRLSCAASG-FSVRN-----NYVSWVRQ...   

                                hcdr3  hcdr3_len  
12         ATTKHGRRIYGVVAFKEWFTYFYMDV         26  
43          ARGWGREQLAPHPSQYYYYYYGMDV         25  
58                              AGNDY

In [ ]:
df_match_check

,Ab_name,Ag_name,Ag_name_details,IC50 [ug/mL],Affinity_Kd [nM],Ag_epitope_restrictions,Ab_heavy_chain_seq,Ab_light_chain_seq,Ag_seq,Ab_structure_method,...,Ag_Lev3_cluster,Ag10_cluster,Ag25_cluster,Ag50_cluster,log_Aff,Aff_op,hc_clean,heavy_aligned_aho,hcdr3,hcdr3_len
0,Sab-2ny3_DC,Ag-2ny3_A,envelope glycoprotein gp120,NaN,815.0,NaN,EVQLVESGAEVKKPGSSVKVSCKASGDTFIRYSFTWVRQAPGQGLE...,DIVMTQSPATLSVSPGERATLSCRASESVSSDLAWYQQKPGQAPRL...,EVVLVNVTENFNMWKNDMVEQMHEDIISLWDQSLKPCVKLTPLCVG...,Crystalized,...,29,19,2,1,2.9112,=,EVQLVESGAEVKKPGSSVKVSCKASGDTFIRYSFTWVRQAPGQGLE...,EVQLVES-GAEVKKPGSSVKVSCKASG-DTFIR-----YSFTWVRQ...,AGVYEGEADEGEYDNNGFLKH,21
1,Sab-1nca_HL,Ag-1nca_N,influenza a subtype n9 neuraminidase,NaN,8.3,NaN,QIQLVQSGPELKKPGETVKISCKASGYTFTNYGMNWVKQAPGKGLK...,DIVMTQSPKFMSTSVGDRVTITCKASQDVSTAVVWYQQKPGQSPKL...,IRDFNNLTKGLCTINSWHIYGKDNAVRIGEDSDVLVTREPYVSCDP...,Crystalized,...,314,210,26,23,0.9191,=,QIQLVQSGPELKKPGETVKISCKASGYTFTNYGMNWVKQAPGKGLK...,QIQLVQS-GPELKKPGETVKISCKASG-YTFTN-----YGMNWVKQ...,ARGEDNFGSLSDY,13
2,Sab-4ypg_HL,Ag-4ypg_D,interferon alpha-2,NaN,0.044,NaN,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYSISWVRQAPGQGLE...,EIVLTQSPGTLSLSPGERATLSCRASQSVSSTYLAWYQQKPGQAPR...,TSCDLPQTHSLGSRRTLMLLAQMRKISLFSCLKDRHDFGFPQEEFG...,Crystalized,...,2321,1350,46,46,-1.3565,=,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYSISWVRQAPGQGLE...,QVQLVQS-GAEVKKPGASVKVSCKASG-YTFTS-----YSISWVRQ...,ARDPIAAGY,9
3,Sab-2ny0_DC,Ag-2ny0_A,envelope glycoprotein gp120,NaN,232.0,NaN,EVQLVESGAEVKKPGSSVKVSCKASGDTFIRYSFTWVRQAPGQGLE...,DIVMTQSPATLSVSPGERATLSCRASESVSSDLAWYQQKPGQAPRL...,EVVLVNVTENFNWCKNDMVEQMHEDIISLWDQSLKPCVKLTPLCVG...,Crystalized,...,278,19,2,1,2.3655,=,EVQLVESGAEVKKPGSSVKVSCKASGDTFIRYSFTWVRQAPGQGLE...,EVQLVES-GAEVKKPGSSVKVSCKASG-DTFIR-----YSFTWVRQ...,AGVYEGEADEGEYDNNGFLKH,21
4,Sab-2r56_IM,Ag-2r56_B,beta-lactoglobulin,NaN,1.3,NaN,QVSLRESGGGLVQPGRSLRLSCTASGFTFRHHGMTWVRQAPGKGLE...,DIVMTQSPSSLSASVGDRVTITCRASQGISSRLAWYQQKPGKAPKL...,TQTMKGLDIQKVAGTWYSLAMAASDISLLDAQSAPLRVYVEELKPT...,Crystalized,...,2314,1348,176,135,0.1139,=,QVSLRESGGGLVQPGRSLRLSCTASGFTFRHHGMTWVRQAPGKGLE...,QVSLRES-GGGLVQPGRSLRLSCTASG-FTFRH-----HGMTWVRQ...,AKAKRVGATGYFDL,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342351,AbCoV-TAU-2310,SARS-CoV-2_Y395C,NaN,NaN,2.8674,RBD_m3,EVQLLESGGGLVQPGGSLRLSCAASGFTFSNYVMSWVRQAPGKGLE...,QSALTQPASVSGSPGQSITISCTGTSSDVGGYDYVSWYQQHPGKAP...,SARS-CoV2_(Y395C),NaN,...,100000,100000,100000,100000,0.4575,=,EVQLLESGGGLVQPGGSLRLSCAASGFTFSNYVMSWVRQAPGKGLE...,EVQLLES-GGGLVQPGGSLRLSCAASG-FTFSN-----YVMSWVRQ...,AKGAAPYYYYYYGMDV,16
342352,AbCoV-TAU-2310,SARS-CoV-2_Y395H,NaN,NaN,2.8717,RBD_m3,EVQLLESGGGLVQPGGSLRLSCAASGFTFSNYVMSWVRQAPGKGLE...,QSALTQPASVSGSPGQSITISCTGTSSDVGGYDYVSWYQQHPGKAP...,SARS-CoV2_(Y395H),NaN,...,100000,100000,100000,100000,0.4581,=,EVQLLESGGGLVQPGGSLRLSCAASGFTFSNYVMSWVRQAPGKGLE...,EVQLLES-GGGLVQPGGSLRLSCAASG-FTFSN-----YVMSWVRQ...,AKGAAPYYYYYYGMDV,16
342353,AbCoV-TAU-2310,SARS-CoV-2_Y395M,NaN,NaN,2.8671,RBD_m3,EVQLLESGGGLVQPGGSLRLSCAASGFTFSNYVMSWVRQAPGKGLE...,QSALTQPASVSGSPGQSITISCTGTSSDVGGYDYVSWYQQHPGKAP...,SARS-CoV2_(Y395M),NaN,...,100000,100000,100000,100000,0.4574,=,EVQLLESGGGLVQPGGSLRLSCAASGFTFSNYVMSWVRQAPGKGLE...,EVQLLES-GGGLVQPGGSLRLSCAASG-FTFSN-----YVMSWVRQ...,AKGAAPYYYYYYGMDV,16
342354,AbCoV-TAU-2310,SARS-CoV-2_Y395S,NaN,NaN,5.2197,RBD_m3,EVQLLESGGGLVQPGGSLRLSCAASGFTFSNYVMSWVRQAPGKGLE...,QSALTQPASVSGSPGQSITISCTGTSSDVGGYDYVSWYQQHPGKAP...,SARS-CoV2_(Y395S),NaN,...,100000,100000,100000,100000,0.7176,=,EVQLLESGGGLVQPGGSLRLSCAASGFTFSNYVMSWVRQAPGKGLE...,EVQLLES-GGGLVQPGGSLRLSCAASG-FTFSN-----YVMSWVRQ...,AKGAAPYYYYYYGMDV,16


In [ ]:
# Check how bad the HCDR3-level conflict is
df_match_check['hcdr3_ag_pair'] = df_match['hcdr3'] + '|' + df_match['Ag_seq']

def parse_kd(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    # Handle censored values: >500 → 500, <0.1 → 0.1
    if val.startswith('>'):
        return float(val[1:])
    if val.startswith('<'):
        return float(val[1:])
    try:
        return float(val)
    except ValueError:
        return np.nan

df_match_check['Affinity_Kd_nM'] = df_match['Affinity_Kd [nM]'].apply(parse_kd)

# on cleaned column
df_match_check['hcdr3_ag_pair'] = df_match_check['hcdr3'] + '|' + df_match_check['Ag_seq']
conflict_check = df_match_check.groupby('hcdr3_ag_pair')['Affinity_Kd_nM'].agg(['count','std','mean'])
conflict_check = conflict_check[conflict_check['count'] > 1]
print(f"HCDR3-Ag pairs with multiple Kd values: {len(conflict_check)}")
print(f"Median std across conflicts: {conflict_check['std'].median():.3f} nM")

HCDR3-Ag pairs with multiple Kd values: 648
Median std across conflicts: 105.026 nM


In [ ]:
# Is the conflict coming from genuinely different measurements or outliers?
print(conflict_check['std'].describe().round(1))
print(f"\nPairs with std > 100 nM: {(conflict_check['std'] > 100).sum()}")
print(f"Pairs with std < 10 nM:  {(conflict_check['std'] < 10).sum()}")

# Are the conflicts from censored values (>500) inflating std?
conflict_rows = df_match_check[df_match_check['hcdr3_ag_pair'].isin(conflict_check.index)]
censored = df_match_check['Affinity_Kd [nM]'].astype(str).str.startswith('>')
print(f"\nCensored values in conflict rows: {censored[conflict_rows.index].sum()}")

count      648.0
mean       433.6
std       3237.4
min          0.0
25%         27.6
50%        105.0
75%        221.1
max      60232.1
Name: std, dtype: float64

Pairs with std > 100 nM: 335
Pairs with std < 10 nM:  103

Censored values in conflict rows: 0


In [ ]:
df_match_check.columns

Index(['Ab_name', 'Ag_name', 'Ag_name_details', 'IC50 [ug/mL]',
       'Affinity_Kd [nM]', 'Ag_epitope_restrictions', 'Ab_heavy_chain_seq',
       'Ab_light_chain_seq', 'Ag_seq', 'Ab_structure_method',
       'Ag_structure_method', 'bound_AbAg_structure_method', 'Ab_PDB_ID',
       'Ag_PDB_ID', 'bound_AbAg_PDB_ID', 'Ab_Lev3_cluster', 'Source', 'escape',
       'log(Kd_ratio)', 'Ab10_cluster', 'Ab25_cluster', 'Ab50_cluster',
       'Ag_Lev3_cluster', 'Ag10_cluster', 'Ag25_cluster', 'Ag50_cluster',
       'log_Aff', 'Aff_op', 'hc_clean', 'heavy_aligned_aho', 'hcdr3',
       'hcdr3_len', 'hcdr3_ag_pair', 'Affinity_Kd_nM'],
      dtype='object')

In [ ]:
# Clean light chain (same as heavy chain cleaning)
df_match_check['lc_clean'] = df_match_check['Ab_light_chain_seq'].apply(clean_seq)

# Now build dedup
df_match_dedup = (df_match_check
    .groupby('hcdr3_ag_pair')
    .agg(
        hcdr3          = ('hcdr3', 'first'),
        Ag_seq         = ('Ag_seq', 'first'),
        Affinity_Kd_nM = ('Affinity_Kd_nM', 'median'),
        vh             = ('hc_clean', 'first'),
        vl             = ('lc_clean', 'first'),
        ag_name        = ('Ag_name', 'first'),
        ab_name        = ('Ab_name', 'first'),
    )
    .reset_index(drop=True)
)
print(f"After dedup: {len(df_match_dedup)} unique HCDR3-Ag pairs")

# Framework variant analysis
def mask_hcdr3(vh, hcdr3):
    if not isinstance(vh, str) or not isinstance(hcdr3, str):
        return vh
    return vh.replace(hcdr3, 'X' * len(hcdr3), 1)

def mask_hcdr3(vh, hcdr3):
    if not isinstance(vh, str) or not isinstance(hcdr3, str):
        return vh
    return vh.replace(hcdr3, 'X' * len(hcdr3), 1)

df_match_dedup['vh_framework'] = df_match_dedup.apply(
    lambda r: mask_hcdr3(r['vh'], r['hcdr3']), axis=1)

# Now run the correct analysis
df_match_dedup['fw_vl_ag'] = (df_match_dedup['vh_framework'] + '|' +
                               df_match_dedup['vl'] + '|' +
                               df_match_dedup['Ag_seq'])

fw_vl_ag_groups = df_match_dedup.groupby('fw_vl_ag')
variants = fw_vl_ag_groups['hcdr3'].nunique()

print(f"Unique framework+VL+Ag groups: {fw_vl_ag_groups.ngroups}")
print(f"\nHCDR3 variants per group:")
print(variants.describe().round(1))
print(f"Groups with >1 HCDR3 variant: {(variants > 1).sum()}")
print(f"Total rows in multi-variant groups: {variants[variants > 1].sum()}")

After dedup: 256650 unique HCDR3-Ag pairs
Unique framework+VL+Ag groups: 252829

HCDR3 variants per group:
count    252829.0
mean          1.0
std           1.3
min           1.0
25%           1.0
50%           1.0
75%           1.0
max         397.0
Name: hcdr3, dtype: float64
Groups with >1 HCDR3 variant: 897
Total rows in multi-variant groups: 4718


In [ ]:
def mask_hcdr3(vh, hcdr3):
    if not isinstance(vh, str) or not isinstance(hcdr3, str):
        return vh
    return vh.replace(hcdr3, 'X' * len(hcdr3), 1)

df_match_dedup['vh_framework'] = df_match_dedup.apply(
    lambda r: mask_hcdr3(r['vh'], r['hcdr3']), axis=1)

# Now run the correct analysis
df_match_dedup['fw_vl_ag'] = (df_match_dedup['vh_framework'] + '|' +
                               df_match_dedup['vl'] + '|' +
                               df_match_dedup['Ag_seq'])

fw_vl_ag_groups = df_match_dedup.groupby('fw_vl_ag')
variants = fw_vl_ag_groups['hcdr3'].nunique()

print(f"Unique framework+VL+Ag groups: {fw_vl_ag_groups.ngroups}")
print(f"\nHCDR3 variants per group:")
print(variants.describe().round(1))
print(f"Groups with >1 HCDR3 variant: {(variants > 1).sum()}")
print(f"Total rows in multi-variant groups: {variants[variants > 1].sum()}")

Unique framework+VL+Ag groups: 252829

HCDR3 variants per group:
count    252829.0
mean          1.0
std           1.3
min           1.0
25%           1.0
50%           1.0
75%           1.0
max         397.0
Name: hcdr3, dtype: float64
Groups with >1 HCDR3 variant: 897
Total rows in multi-variant groups: 4718


In [ ]:
multi_variant_groups = variants[variants > 1].index
df_finetune = df_match_dedup[
    df_match_dedup['fw_vl_ag'].isin(multi_variant_groups)
].copy()

print(f"Stage 2 fine-tuning rows: {len(df_finetune)}")
print(f"With Kd labels: {df_finetune['Affinity_Kd_nM'].notna().sum()}")
print(f"Unique frameworks: {df_finetune['vh_framework'].nunique()}")
print(f"Unique antigens:   {df_finetune['Ag_seq'].nunique()}")

Stage 2 fine-tuning rows: 4718
With Kd labels: 4607
Unique frameworks: 844
Unique antigens:   56


In [ ]:
# 1. Full deduped dataset — for Stage 1 InfoNCE pretraining
df_match_dedup.to_csv(f'{PROJECT_DIR}/abrank_stage1.csv', index=False)

# 2. Multi-variant groups only — for Stage 2 MSE fine-tuning
df_finetune.to_csv(f'{PROJECT_DIR}/abrank_stage2.csv', index=False)

# 3. ANARCI cache — so you never have to rerun alignment
import json
with open(f'{PROJECT_DIR}/anarci_heavy_aho.json', 'w') as f:
    json.dump(vh_aho_dict, f)

print('Saved: abrank_stage1.csv, abrank_stage2.csv, anarci_heavy_aho.json')

Saved: abrank_stage1.csv, abrank_stage2.csv, anarci_heavy_aho.json


### 1.1b — Build `sabdab_pretrain` for binding scorer training

In [ ]:
df_stage1 = pd.read_csv(f'{PROJECT_DIR}/abrank_stage1.csv')
df_stage2 = pd.read_csv(f'{PROJECT_DIR}/abrank_stage2.csv')

sabdab_pretrain = df_stage1  # 256k rows, all unique HCDR3-Ag pairs
sabdab_finetune = df_stage2  # 4,718 rows, comparative HCDR3 signal only

# Exclude clinical mAbs from Stage 2
clinical_vh_set = set(df[COL_VH].tolist())
clinical_vl_set = set(df[COL_VL].tolist())
sabdab_finetune = sabdab_finetune[
    ~(sabdab_finetune['vh'].isin(clinical_vh_set) &
      sabdab_finetune['vl'].isin(clinical_vl_set))
].reset_index(drop=True)

print(f'Stage 1 (pretrain): {len(sabdab_pretrain)} pairs')
print(f'Stage 2 (finetune): {len(sabdab_finetune)} pairs')

Stage 1 (pretrain): 256650 pairs
Stage 2 (finetune): 4717 pairs


In [ ]:
sabdab_pretrain = df_stage1
sabdab_finetune = df_stage2  # 1 clinical ab exclude from rank dataset

print(f'Stage 1 (pretrain): {len(sabdab_pretrain)} pairs')
print(f'Stage 2 (finetune): {len(sabdab_finetune)} pairs')

Stage 1 (pretrain): 256650 pairs
Stage 2 (finetune): 4718 pairs


### 1.1d — ESM2 antigen embeddings (mean-pooled + per-residue)

Two embedding formats are needed:
- **Mean-pooled** `(N, 1280)` — for InfoNCE contrastive pretraining
- **Per-residue padded** `(N, ANTIGEN_MAX_LEN, 1280)` — for cross-attention scoring and ODE guidance

In [ ]:
# ── Load ESM2-650M ────────────────────────────────────────────────────────────
print('Loading ESM2-650M...')
esm2_model, esm2_alphabet = esm.pretrained.esm2_t33_650M_UR50D()
esm2_model = esm2_model.eval().to(DEVICE)
batch_converter = esm2_alphabet.get_batch_converter()
print('ESM2 loaded.')

Loading ESM2-650M...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
ESM2 loaded.


add antigen information for the clinical antibody

In [ ]:
import requests

def fetch_uniprot_sequence(gene_name, organism='human'):
    """Fetch canonical protein sequence from UniProt by gene name."""
    url = 'https://rest.uniprot.org/uniprotkb/search'
    params = {
        'query': f'gene:{gene_name} AND organism_id:9606 AND reviewed:true',
        'format': 'fasta',
        'size': 1
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200 and r.text:
            lines = r.text.strip().split('\n')
            seq = ''.join(l for l in lines if not l.startswith('>'))
            return seq if seq else None
    except Exception as e:
        print(f'UniProt fetch failed for {gene_name}: {e}')
    return None



In [ ]:
CLINICAL_TARGETS = {
    'abagovomab':      'CA125',       # MUC16
    'abituzumab':      'ITGAV',       # CD51
    'abrezekimab':     'IL13',
    'abrilumab':       'ITGA4',       # integrin α4β7
    'adalimumab':      'TNF',
    'aducanumab':      'APP',         # amyloid beta
    'alemtuzumab':     'CD52',
    'alirocumab':      'PCSK9',
    'amatuximab':      'MSLN',        # mesothelin
    'andecaliximab':   'MMP9',        # gelatinase B
    'anetumab':        'MSLN',        # mesothelin
    'anifrolumab':     'IFNAR1',      # IFN-α/β receptor
    'anrukinzumab':    'IL13',
    'atezolizumab':    'CD274',       # PD-L1
    'avelumab':        'CD274',       # PD-L1
    'bapineuzumab':    'APP',         # β-amyloid
    'basiliximab':     'IL2RA',       # CD25
    'bavituximab':     'PTDSS1',      # phosphatidylserine
    'belantamab':      'TNFRSF17',    # BCMA
    'belimumab':       'TNFSF13B',    # BAFF
    'bemarituzumab':   'FGFR2',
    'benralizumab':    'IL5RA',       # CD125
    'bevacizumab':     'VEGFA',
    'bezlotoxumab':    'CDT',         # C. diff toxin B
    'bimagrumab':      'ACVR2B',
    'bimekizumab':     'IL17A',
    'bleselumab':      'CD40',
    'blosozumab':      'SOST',
    'bococizumab':     'PCSK9',
    'brazikumab':      'IL23A',       # IL-23
    'brentuximab':     'TNFRSF8',    # CD30
    'briakinumab':     'IL12A',       # IL-12/23
    'brodalumab':      'IL17RA',
    'brontictuzumab':  'NOTCH1',
    'budigalimab':     'PDCD1',       # PD-1
    'burosumab':       'FGF23',
    'cabiralizumab':   'CSF1R',
    'camidanlumab':    'IL2RA',       # CD25
    'camrelizumab':    'PDCD1',       # PD-1
    'carlumab':        'CCL2',        # MCP-1
    'cemiplimab':      'PDCD1',       # PD-1
    'certolizumab':    'TNF',
    'cetrelimab':      'PDCD1',       # PD-1
    'cetuximab':       'EGFR',
    'cixutumumab':     'IGF1R',
    'clazakizumab':    'IL6',
    'codrituzumab':    'GPC3',
    'coltuximab':      'CD19',
    'concizumab':      'TFPI',
    'crenezumab':      'APP',         # β-amyloid
    'crizanlizumab':   'SELP',        # P-selectin
    'cusatuzumab':     'CD70',
    'dacetuzumab':     'CD40',
    'daclizumab':      'IL2RA',       # CD25
    'dalotuzumab':     'IGF1R',
    'daratumumab':     'CD38',
    'denosumab':       'TNFSF11',     # RANKL
    'depatuxizumab':   'EGFR',
    'dinutuximab':     'B4GALNT1',    # GD2 ganglioside
    'domagrozumab':    'MSTN',        # GDF-8
    'dostarlimab':     'PDCD1',       # PD-1
    'duligotuzumab':   'ERBB3',
    'dupilumab':       'IL4R',
    'durvalumab':      'CD274',       # PD-L1
    'eculizumab':      'C5',
    'efalizumab':      'ITGAL',       # LFA-1/CD11a
    'eldelumab':       'CXCL10',
    'elezanumab':      'RGMA',
    'elotuzumab':      'SLAMF7',
    'emactuzumab':     'CSF1R',
    'emapalumab':      'IFNG',        # IFN-γ
    'emibetuzumab':    'MET',         # HGFR
    'enavatuzumab':    'TNFRSF12A',   # TWEAK receptor
    'enokizumab':      'IL9',
    'epratuzumab':     'CD22',
    'eptinezumab':     'CALCA',       # CGRP
    'erenumab':        'CALCRL',      # CGRP receptor
    'etaracizumab':    'ITGAV',       # integrin αvβ3
    'etrolizumab':     'ITGB7',
    'evinacumab':      'ANGPTL3',
    'evolocumab':      'PCSK9',
    'farletuzumab':    'FOLR1',
    'fasinumab':       'NGF',
    'fezakinumab':     'IL22',
    'ficlatuzumab':    'HGF',
    'figitumumab':     'IGF1R',
    'fletikumab':      'IL20',
    'foralumab':       'CD3E',
    'fremanezumab':    'CALCA',       # CGRP
    'fresolimumab':    'TGFB1',
    'fulranumab':      'NGF',
    'galcanezumab':    'CALCA',       # CGRP
    'galiximab':       'CD80',
    'ganitumab':       'IGF1R',
    'gantenerumab':    'APP',         # β-amyloid
    'gatipotuzumab':   'MUC1',
    'gemtuzumab':      'CD33',
    'gevokizumab':     'IL1B',
    'gimsilumab':      'CSF2',        # GM-CSF
    'girentuximab':    'CA9',
    'glembatumumab':   'GPNMB',
    'golimumab':       'TNF',
    'guselkumab':      'IL23A',
    'ianalumab':       'TNFRSF13C',   # BAFF-R
    'ibalizumab':      'CD4',
    'icrucumab':       'VEGFR1',
    'imgatuzumab':     'EGFR',
    'inclacumab':      'SELP',        # P-selectin
    'inebilizumab':    'CD19',
    'infliximab':      'TNF',
    'inotuzumab':      'CD22',
    'intetumumab':     'ITGAV',
    'ipilimumab':      'CTLA4',
    'isatuximab':      'CD38',
    'iscalimab':       'CD40',
    'itolizumab':      'CD6',
    'ixekizumab':      'IL17A',
    'ladiratuzumab':   'LIV1',        # SLC39A6
    'lampalizumab':    'CFD',         # complement factor D
    'lanadelumab':     'KLKB1',       # plasma kallikrein
    'landogrozumab':   'MSTN',        # GDF-8
    'lebrikizumab':    'IL13',
    'lenzilumab':      'CSF2',        # GM-CSF
    'lexatumumab':     'TNFRSF10B',   # TRAIL-R2
    'ligelizumab':     'FCER1A',      # IgE receptor
    'lintuzumab':      'CD33',
    'lirilumab':       'KIR2DL1',
    'loncastuximab':   'CD19',
    'lorvotuzumab':    'CD56',
    'lucatumumab':     'CD40',
    'lumiliximab':     'CD23',
    'lumretuzumab':    'ERBB3',
    'margetuximab':    'ERBB2',       # HER2
    'matuzumab':       'EGFR',
    'mavrilimumab':    'CSF2RA',      # GM-CSFR
    'mepolizumab':     'IL5',
    'milatuzumab':     'CD74',
    'mirikizumab':     'IL23A',
    'mirvetuximab':    'FOLR1',
    'mitazalimab':     'CD40',
    'mogamulizumab':   'CCR4',
    'monalizumab':     'KLRC1',       # NKG2A
    'mosunetuzumab':   'CD20',
    'motavizumab':     'F',           # RSV F protein
    'muromonab':       'CD247',       # CD3
    'natalizumab':     'ITGA4',
    'necitumumab':     'EGFR',
    'nesvacumab':      'ANGPT2',
    'nimotuzumab':     'EGFR',
    'nivolumab':       'PDCD1',       # PD-1
    'obexelimab':      'FCER2',       # CD23
    'obiltoxaximab':   'PA',          # anthrax protective antigen
    'obinutuzumab':    'MS4A1',       # CD20
    'ocrelizumab':     'MS4A1',       # CD20
    'ofatumumab':      'MS4A1',       # CD20
    'olaratumab':      'PDGFRA',
    'oleclumab':       'NT5E',        # CD73
    'olokizumab':      'IL6',
    'omalizumab':      'IGER',        # IgE
    'omburtamab':      'B7H3',        # CD276
    'onartuzumab':     'MET',
    'opicinumab':      'MOG',
    'orticumab':       'OXLDL',       # oxidized LDL
    'osocimab':        'F11',         # FXIa
    'otelixizumab':    'CD247',       # CD3
    'otlertuzumab':    'CD37',
    'ozanezumab':      'MAG',
    'palivizumab':     'F',           # RSV F protein
    'pamrevlumab':     'CCN2',        # CTGF
    'panitumumab':     'EGFR',
    'panobacumab':     'PCRV',        # P. aeruginosa
    'parsatuzumab':    'EGFL7',
    'patritumab':      'ERBB3',
    'pembrolizumab':   'PDCD1',       # PD-1
    'pertuzumab':      'ERBB2',
    'pidilizumab':     'PDCD1',       # PD-1
    'pinatuzumab':     'CD22',
    'plozalizumab':    'CCR2',
    'polatuzumab':     'CD79B',
    'ponezumab':       'APP',         # amyloid beta
    'prasinezumab':    'SNCA',        # alpha-synuclein
    'prezalumab':      'TNFRSF13C',   # BAFF-R
    'prolgolimab':     'PDCD1',       # PD-1
    'quilizumab':      'IGER',        # IgE
    'racotumomab':     'NGCGM3',      # N-glycolyl GM3
    'radretumab':      'FN1',         # fibronectin extra domain B
    'ramucirumab':     'KDR',         # VEGFR2
    'ranibizumab':     'VEGFA',
    'relatlimab':      'LAG3',
    'reslizumab':      'IL5',
    'rilotumumab':     'HGF',
    'rinucumab':       'PDGFRB',
    'risankizumab':    'IL23A',
    'rituximab':       'MS4A1',       # CD20
    'robatumumab':     'IGF1R',
    'romosozumab':     'SOST',
    'rontalizumab':    'IFNA1',       # IFN-α
    'rovalpituzumab':  'DLL3',
    'rozanolixizumab': 'FCGRT',       # FcRn
    'sarilumab':       'IL6R',
    'satralizumab':    'IL6R',
    'secukinumab':     'IL17A',
    'selicrelumab':    'CD40',
    'seribantumab':    'ERBB3',
    'setrusumab':      'SOST',
    'sifalimumab':     'IFNA1',       # IFN-α
    'siltuximab':      'IL6',
    'simtuzumab':      'LOXL2',
    'sintilimab':      'PDCD1',       # PD-1
    'sirukumab':       'IL6',
    'solanezumab':     'APP',         # amyloid beta
    'spartalizumab':   'PDCD1',       # PD-1
    'sutimlimab':      'C1S',         # complement C1s
    'tabalumab':       'TNFSF13B',    # BAFF
    'tanezumab':       'NGF',
    'tarextumab':      'NOTCH2',
    'tavolimab':       'TNFRSF14',    # HVEM
    'telisotuzumab':   'MET',
    'teplizumab':      'CD247',       # CD3
    'teprotumumab':    'IGF1R',
    'tezepelumab':     'TSLP',
    'tigatuzumab':     'TNFRSF10B',   # DR5
    'tildrakizumab':   'IL23A',
    'tislelizumab':    'PDCD1',       # PD-1
    'tisotumab':       'F3',          # tissue factor
    'tocilizumab':     'IL6R',
    'toripalimab':     'PDCD1',       # PD-1
    'tovetumab':       'PDGFRA',
    'tralokinumab':    'IL13',
    'trastuzumab':     'ERBB2',
    'tregalizumab':    'CD4',
    'tremelimumab':    'CTLA4',
    'ublituximab':     'MS4A1',       # CD20
    'urelumab':        'CD137',       # 4-1BB
    'ustekinumab':     'IL12A',       # IL-12/23
    'utomilumab':      'TNFRSF9',     # CD137/4-1BB
    'vadastuximab':    'CD33',
    'varlilumab':      'CD27',
    'vatelizumab':     'ITGA2',       # CD49b
    'vedolizumab':     'ITGA4',       # integrin α4β7
    'veltuzumab':      'MS4A1',       # CD20
    'visilizumab':     'CD247',       # CD3
    'xentuzumab':      'IGF1',
    'zalutumumab':     'EGFR',
    'zanolimumab':     'CD4',
    'zolbetuximab':    'CLDN18',      # claudin-18.2
}

print(f'Total antibodies: {len(CLINICAL_TARGETS)}')

Total antibodies: 246


In [ ]:
def fetch_uniprot_sequence(gene_name, organism_id=9606):
    url = 'https://rest.uniprot.org/uniprotkb/search'
    params = {
        'query': f'gene:{gene_name} AND organism_id:{organism_id} AND reviewed:true',
        'format': 'fasta',
        'size': 1
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200 and r.text.strip():
            lines = r.text.strip().split('\n')
            return ''.join(l for l in lines if not l.startswith('>'))
    except Exception as e:
        print(f'Failed {gene_name}: {e}')
    return None

antigen_seqs = {}
for ab, gene in CLINICAL_TARGETS.items():
    seq = fetch_uniprot_sequence(gene)
    antigen_seqs[ab] = seq
    status = f'{len(seq)} aa' if seq else 'NOT FOUND'
    print(f'{ab:25s} -> {gene:15s}: {status}')

abagovomab                -> CA125          : 14507 aa
abituzumab                -> ITGAV          : 1048 aa
abrezekimab               -> IL13           : 146 aa
abrilumab                 -> ITGA4          : 1032 aa
adalimumab                -> TNF            : 233 aa
aducanumab                -> APP            : 770 aa
alemtuzumab               -> CD52           : 61 aa
alirocumab                -> PCSK9          : 692 aa
amatuximab                -> MSLN           : 630 aa
andecaliximab             -> MMP9           : 707 aa
anetumab                  -> MSLN           : 630 aa
anifrolumab               -> IFNAR1         : 557 aa
anrukinzumab              -> IL13           : 146 aa
atezolizumab              -> CD274          : 290 aa
avelumab                  -> CD274          : 290 aa
bapineuzumab              -> APP            : 770 aa
basiliximab               -> IL2RA          : 272 aa
bavituximab               -> PTDSS1         : 473 aa
belantamab                -> TNFRSF17      

In [ ]:
# ── Build antigen_df from UniProt sequences ───────────────────────────────────
antigen_df = pd.DataFrame({
    'antibody_name':    df[COL_NAME].tolist(),
    'antigen_gene':     [CLINICAL_TARGETS.get(name.lower(), None)
                         for name in df[COL_NAME].tolist()],
    'antigen_sequence': [antigen_seqs.get(name.lower(), None)
                         for name in df[COL_NAME].tolist()],
})

n_found  = antigen_df['antigen_sequence'].notna().sum()
n_target = antigen_df['antigen_gene'].notna().sum()
print(f'Clinical mAbs with known target:     {n_target} / {len(df)}')
print(f'Clinical mAbs with antigen sequence: {n_found} / {len(df)}')

missing = antigen_df[antigen_df['antigen_sequence'].isna()]
if len(missing) > 0:
    print(f'\nMissing sequences ({len(missing)}):')
    print(missing[['antibody_name', 'antigen_gene']].to_string(index=False))

Clinical mAbs with known target:     246 / 246
Clinical mAbs with antigen sequence: 240 / 246

Missing sequences (6):
antibody_name antigen_gene
 bezlotoxumab          CDT
 lorvotuzumab         CD56
  lumiliximab         CD23
    orticumab        OXLDL
  panobacumab         PCRV
  racotumomab       NGCGM3


In [ ]:
# Fix the 6 problem entries
CLINICAL_TARGETS['bezlotoxumab'] = 'TcdB'      # C. diff toxin B - not a human gene, skip
CLINICAL_TARGETS['lorvotuzumab'] = 'NCAM1'     # CD56 = NCAM1 in UniProt
CLINICAL_TARGETS['lumiliximab']  = 'FCER2'     # CD23 = FCER2 in UniProt
CLINICAL_TARGETS['orticumab']    = 'OLR1'      # oxidized LDL receptor
CLINICAL_TARGETS['panobacumab']  = None         # P. aeruginosa antigen, no human UniProt entry
CLINICAL_TARGETS['racotumomab']  = None         # N-glycolyl GM3 ganglioside, no UniProt entry

# Re-fetch only the fixable ones
fixable = {k: v for k, v in CLINICAL_TARGETS.items()
           if k in ['lorvotuzumab', 'lumiliximab', 'orticumab'] and v}

for ab, gene in fixable.items():
    seq = fetch_uniprot_sequence(gene)
    antigen_seqs[ab] = seq
    print(f'{ab:20s} -> {gene}: {"found" if seq else "NOT FOUND"} '
          f'({len(seq) if seq else 0} aa)')

# for bezlotoxumab?, panobacumab, racotumomab: x target protein:ODE guidance will fall back to λ=0

lorvotuzumab         -> NCAM1: found (858 aa)
lumiliximab          -> FCER2: found (321 aa)
orticumab            -> OLR1: found (273 aa)


In [ ]:
def fetch_uniprot_sequence_by_organism(gene_name, organism_id):
    url = 'https://rest.uniprot.org/uniprotkb/search'
    params = {
        'query': f'gene:{gene_name} AND organism_id:{organism_id} AND reviewed:true',
        'format': 'fasta',
        'size': 1
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200 and r.text.strip():
            lines = r.text.strip().split('\n')
            return ''.join(l for l in lines if not l.startswith('>'))
    except Exception as e:
        print(f'Failed {gene_name}: {e}')
    return None

# C. difficile organism ID = 272563 (reference strain 630)
seq = fetch_uniprot_sequence_by_organism('tcdB', 272563)
if seq:
    antigen_seqs['bezlotoxumab'] = seq
    print(f'bezlotoxumab -> TcdB: found ({len(seq)} aa)')
else:
    # Fallback: fetch by UniProt accession directly (Q9BKG9 = TcdB)
    r = requests.get('https://rest.uniprot.org/uniprotkb/Q9BKG9.fasta', timeout=10)
    if r.status_code == 200:
        seq = ''.join(l for l in r.text.split('\n') if not l.startswith('>'))
        antigen_seqs['bezlotoxumab'] = seq
        print(f'bezlotoxumab -> TcdB (accession): found ({len(seq)} aa)')

bezlotoxumab -> TcdB (accession): found (171 aa)


In [ ]:
# Rebuild antigen_df with updated antigen_seqs after fixes
antigen_df['antigen_sequence'] = [
    antigen_seqs.get(name.lower(), None)
    for name in df[COL_NAME].tolist()
]

# Add antigen info to df and save
df['antigen_gene']     = antigen_df['antigen_gene'].values
df['antigen_sequence'] = antigen_df['antigen_sequence'].values

df.to_csv(f'{PROJECT_DIR}/clinical_mabs_with_antigens.csv', index=False)
print(f'Saved clinical mAbs with antigen sequences.')
print(f'Coverage: {df["antigen_sequence"].notna().sum()} / {len(df)}')


Saved clinical mAbs with antigen sequences.
Coverage: 244 / 246


In [ ]:
@torch.no_grad()
def embed_sequence_esm2(seq, max_len=ANTIGEN_MAX_LEN):
    """
    Embed antigen sequence with ESM2-650M.
    Returns:
        mean  : (1280,)             mean-pooled over residues
        resid : (seq_len, 1280)     per-residue (truncated to max_len)
    """
    seq = seq[:max_len]
    _, _, tokens = batch_converter([('antigen', seq)])
    tokens = tokens.to(DEVICE)
    out  = esm2_model(tokens, repr_layers=[33], return_contacts=False)
    reps = out['representations'][33][0, 1:-1, :]   # strip BOS/EOS
    mean  = reps.mean(dim=0).cpu().numpy()
    resid = reps.cpu().numpy()
    return mean, resid

# ── Cache paths ───────────────────────────────────────────────────────────────
MEAN_CACHE       = f'{EVAL_DIR}/antigen_esm2_embs.npy'
RESID_CACHE      = f'{EVAL_DIR}/antigen_esm2_resid.npy'
MASK_CACHE       = f'{EVAL_DIR}/antigen_valid_mask.npy'
RESID_PAD_CACHE  = f'{EVAL_DIR}/antigen_resid_pad_mask.npy'
SAB_MEAN_CACHE   = f'{EVAL_DIR}/sabdab_ag_embs.npy'
SAB_CDR3_CACHE   = f'{EVAL_DIR}/sabdab_cdr3_embs.npy'






In [ ]:
# ── Embed antigens for clinical mAbs (for ODE guidance) ──────────────────────
# One row per clinical mAb in df, matched by antibody name
antigen_df = pd.DataFrame({
    'antibody_name':    df[COL_NAME].tolist(),
    'antigen_gene':     [CLINICAL_TARGETS.get(name.lower(), None)
                         for name in df[COL_NAME].tolist()],
    'antigen_sequence': [antigen_seqs.get(name.lower(), None)
                         for name in df[COL_NAME].tolist()],
})

n_found = antigen_df['antigen_sequence'].notna().sum()
n_target = antigen_df['antigen_gene'].notna().sum()
print(f'Clinical mAbs with known target:    {n_target} / {len(df)}')
print(f'Clinical mAbs with antigen sequence: {n_found} / {len(df)}')

# Show any gaps
missing = antigen_df[antigen_df['antigen_sequence'].isna()]
if len(missing) > 0:
    print(f'\nMissing sequences ({len(missing)}):')
    print(missing[['antibody_name', 'antigen_gene']].to_string(index=False))

In [ ]:
if (os.path.exists(MEAN_CACHE) and os.path.exists(RESID_CACHE)
        and os.path.exists(MASK_CACHE)):
    mean_embs      = np.load(MEAN_CACHE)
    resid_embs     = np.load(RESID_CACHE)
    valid_mask     = np.load(MASK_CACHE).astype(bool)
    resid_pad_mask = np.load(RESID_PAD_CACHE).astype(bool)
    print(f'Loaded clinical antigen embeddings from cache.')
    print(f'  mean:  {mean_embs.shape}')
    print(f'  resid: {resid_embs.shape}')
else:
    N = len(antigen_df)
    mean_embs      = np.zeros((N, ESM2_DIM), dtype=np.float32)
    resid_embs     = np.zeros((N, ANTIGEN_MAX_LEN, ESM2_DIM), dtype=np.float32)
    resid_pad_mask = np.ones((N, ANTIGEN_MAX_LEN), dtype=bool)   # True = padding
    valid_mask     = np.zeros(N, dtype=bool)

    print(f'Embedding {N} clinical mAb antigens with ESM2...')
    for i, row in antigen_df.iterrows():
        seq = row['antigen_sequence']
        if not isinstance(seq, str) or len(seq) == 0:
            continue
        try:
            mean_v, resid_v = embed_sequence_esm2(seq)
            L = min(len(resid_v), ANTIGEN_MAX_LEN)
            mean_embs[i]          = mean_v
            resid_embs[i, :L]     = resid_v[:L]
            resid_pad_mask[i, :L] = False      # real positions
            valid_mask[i]         = True
        except Exception as e:
            print(f'  ✗ {row["antibody_name"]}: {e}')

    np.save(MEAN_CACHE,      mean_embs)
    np.save(RESID_CACHE,     resid_embs)
    np.save(MASK_CACHE,      valid_mask)
    np.save(RESID_PAD_CACHE, resid_pad_mask)
    print(f'Saved. Coverage: {valid_mask.sum()} / {N}')

antigen_df['esm2_valid'] = valid_mask
print(f'\nAntigen coverage: {valid_mask.sum()} / {len(antigen_df)}')

In [ ]:
sabdab_pretrain.head()

,hcdr3,Ag_seq,Affinity_Kd_nM,vh,vl,ag_name,ab_name,vh_framework,fw_vl_ag
0,AAAIIASAAPDY,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,NaN,QVQLVQSGPEVKKPGTSVKVSCKASGFTFSSSAVQWVRQARGQRLE...,EIVLTQSPATLSLSPGERATLSCRASQSVSSYLAWYQQKPGQAPRL...,SARS-CoV-2,AbCoV-CS143,QVQLVQSGPEVKKPGTSVKVSCKASGFTFSSSAVQWVRQARGQRLE...,QVQLVQSGPEVKKPGTSVKVSCKASGFTFSSSAVQWVRQARGQRLE...
1,AADGYRGTYSF,TQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTN...,79.19,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKL...,HER2,AbHER-528,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...
2,AADLGILWFGDLRKSEP,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,NaN,EVQLVESGGGLVQPGGSLRLSCAASGFIFNRYWMTWVRQAPGKGLE...,DIQMTQSPFSLSASVGDRVSITCRASQGISNSLAWYQQKPGKAPKL...,SARS-CoV-2,AbCoV-PDI-307,EVQLVESGGGLVQPGGSLRLSCAASGFIFNRYWMTWVRQAPGKGLE...,EVQLVESGGGLVQPGGSLRLSCAASGFIFNRYWMTWVRQAPGKGLE...
3,AADPAMVRGVIMGNWFDP,SARS-CoV2_(A351G),NaN,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQIISTYLNWYQQKPGKAPKL...,SARS-CoV-2_A351G,AbBloom-BD55-1499,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...
4,AADPAMVRGVIMGNWFDP,SARS-CoV2_(A351Q),NaN,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQIISTYLNWYQQKPGKAPKL...,SARS-CoV-2_A351Q,AbBloom-BD55-1499,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...


In [ ]:
print(f"Total rows: {len(sabdab_pretrain)}")
print(f"Null ag:    {sabdab_pretrain['Ag_seq'].isna().sum()}")
print(f"Empty ag:   {(sabdab_pretrain['Ag_seq'] == '').sum()}")
print(f"Valid ag:   {sabdab_pretrain['Ag_seq'].notna().sum()}")

# Check if it's concentrated in certain antigens
print(sabdab_pretrain[sabdab_pretrain['Ag_seq'].isna()][['ab_name','ag_name']].head(10))

Total rows: 256650
Null ag:    0
Empty ag:   0
Valid ag:   256650
Empty DataFrame
Columns: [ab_name, ag_name]
Index: []


In [ ]:
sabdab_pretrain = sabdab_pretrain.rename(columns={'Ag_seq': 'ag'})
sabdab_finetune = sabdab_finetune.rename(columns={'Ag_seq': 'ag'})

# Save cleaned versions
sabdab_pretrain.to_csv(f'{PROJECT_DIR}/abrank_stage1.csv', index=False)
sabdab_finetune.to_csv(f'{PROJECT_DIR}/abrank_stage2.csv', index=False)
print('Renamed Ag_seq -> ag and saved.')

Renamed Ag_seq -> ag and saved.


# rerun

In [ ]:
sabdab_pretrain = pd.read_csv(f'{PROJECT_DIR}/abrank_stage1.csv')
sabdab_finetune = pd.read_csv(f'{PROJECT_DIR}/abrank_stage2.csv')

In [ ]:
print(sabdab_pretrain.columns.tolist())
print(sabdab_pretrain['ag'].notna().sum())
print(sabdab_pretrain['ag'].iloc[0][:50])  # sanity check first sequence

['hcdr3', 'ag', 'Affinity_Kd_nM', 'vh', 'vl', 'ag_name', 'ab_name', 'vh_framework', 'fw_vl_ag']
256650
MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHS


In [ ]:
sabdab_pretrain.head()

,hcdr3,ag,Affinity_Kd_nM,vh,vl,ag_name,ab_name,vh_framework,fw_vl_ag
0,AAAIIASAAPDY,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,NaN,QVQLVQSGPEVKKPGTSVKVSCKASGFTFSSSAVQWVRQARGQRLE...,EIVLTQSPATLSLSPGERATLSCRASQSVSSYLAWYQQKPGQAPRL...,SARS-CoV-2,AbCoV-CS143,QVQLVQSGPEVKKPGTSVKVSCKASGFTFSSSAVQWVRQARGQRLE...,QVQLVQSGPEVKKPGTSVKVSCKASGFTFSSSAVQWVRQARGQRLE...
1,AADGYRGTYSF,TQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTN...,79.19,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKL...,HER2,AbHER-528,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...
2,AADLGILWFGDLRKSEP,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,NaN,EVQLVESGGGLVQPGGSLRLSCAASGFIFNRYWMTWVRQAPGKGLE...,DIQMTQSPFSLSASVGDRVSITCRASQGISNSLAWYQQKPGKAPKL...,SARS-CoV-2,AbCoV-PDI-307,EVQLVESGGGLVQPGGSLRLSCAASGFIFNRYWMTWVRQAPGKGLE...,EVQLVESGGGLVQPGGSLRLSCAASGFIFNRYWMTWVRQAPGKGLE...
3,AADPAMVRGVIMGNWFDP,SARS-CoV2_(A351G),NaN,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQIISTYLNWYQQKPGKAPKL...,SARS-CoV-2_A351G,AbBloom-BD55-1499,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...
4,AADPAMVRGVIMGNWFDP,SARS-CoV2_(A351Q),NaN,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQIISTYLNWYQQKPGKAPKL...,SARS-CoV-2_A351Q,AbBloom-BD55-1499,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...,QVQLQQWGAGLLKPSETLSLTCAVYGGSFTGYYWSWIRQPPGKGLE...


In [ ]:
# Flag rows where 'ag' looks like a name not a sequence
def is_valid_seq(s):
    if not isinstance(s, str) :
        return False
    valid_aa = set('ACDEFGHIKLMNPQRSTVWY')
    return sum(c in valid_aa for c in s.upper()) / len(s) > 0.9

valid_ag = sabdab_pretrain['ag'].apply(is_valid_seq)
print(f'Valid sequences: {valid_ag.sum()} / {len(sabdab_pretrain)}')
print(f'Invalid:         {(~valid_ag).sum()}')
print(f'\nSample invalid:')
print(sabdab_pretrain[~valid_ag]['ag'].value_counts().head(10))

Valid sequences: 70999 / 256650
Invalid:         185651

Sample invalid:
ag
SARS-CoV2_(C360N)    531
SARS-CoV2_(C360E)    524
SARS-CoV2_(F346N)    502
SARS-CoV2_(C360P)    477
SARS-CoV2_(C360D)    441
SARS-CoV2_(Y364P)    439
SARS-CoV2_(Y364C)    437
SARS-CoV2_(F346H)    427
SARS-CoV2_(K355S)    422
SARS-CoV2_(S513F)    420
Name: count, dtype: int64


In [ ]:
# filter out spike seq
sabdab_pretrain = sabdab_pretrain[valid_ag].reset_index(drop=True)
sabdab_finetune = sabdab_finetune[
    sabdab_finetune['ag'].apply(is_valid_seq)
].reset_index(drop=True)

In [ ]:
#sabdab_pretrain.head()
len(sabdab_pretrain)

70999

In [ ]:

sabdab_pretrain.to_csv(f'{PROJECT_DIR}/abrank_stage1_val.csv', index=False)
sabdab_finetune.to_csv(f'{PROJECT_DIR}/abrank_stage2_val.csv', index=False)
print(f'pretrain: {len(sabdab_pretrain)}  finetune: {len(sabdab_finetune)}')

pretrain: 70999  finetune: 4610


rerun

In [ ]:
sabdab_pretrain = pd.read_csv(f'{PROJECT_DIR}/abrank_stage1_val.csv')
sabdab_finetune = pd.read_csv(f'{PROJECT_DIR}/abrank_stage2_val.csv')

In [ ]:
len(sabdab_pretrain)

70999

In [ ]:
# ── Embed AbRank antigens (mean-pooled only, for contrastive pretraining) ─────
CHECKPOINT_INTERVAL = 700

if os.path.exists(SAB_MEAN_CACHE):
    sab_ag_embs = np.load(SAB_MEAN_CACHE)
    print(f'Loaded AbRank antigen embs from cache: {sab_ag_embs.shape}')
else:
    N = len(sabdab_pretrain)

    # Resume from partial checkpoint if it exists
    PARTIAL_CACHE = SAB_MEAN_CACHE.replace('.npy', '_partial.npy')
    if os.path.exists(PARTIAL_CACHE):
        sab_ag_embs = np.load(PARTIAL_CACHE)
        # Find last completed row
        start_i = int(np.where(sab_ag_embs.sum(axis=1) != 0)[0].max()) + 1
        print(f'Resuming from row {start_i}...')
    else:
        sab_ag_embs = np.zeros((N, ESM2_DIM), dtype=np.float32)
        start_i = 0
        print(f'Embedding {N} AbRank antigens with ESM2 (mean-pooled)...')

    for i, row in sabdab_pretrain.iloc[start_i:].iterrows():
        try:
            mean_v, _ = embed_sequence_esm2(row['ag'])
            sab_ag_embs[i] = mean_v
        except Exception as e:
            print(f'  ✗ row {i}: {e}')
        if (i + 1) % CHECKPOINT_INTERVAL == 0:
            np.save(PARTIAL_CACHE, sab_ag_embs)
            print(f'  {i+1}/{N} — checkpoint saved')

    np.save(SAB_MEAN_CACHE, sab_ag_embs)
    if os.path.exists(PARTIAL_CACHE):
        os.remove(PARTIAL_CACHE)
    print(f'Saved: {sab_ag_embs.shape}')

In [ ]:
# ── Re-run failed rows in sab_ag_embs ─────────────────────────────────────────

# Find rows that are still all-zeros (failed/skipped)
failed_indices = np.where(sab_ag_embs.sum(axis=1) == 0)[0]
print(f"Found {len(failed_indices)} failed rows to re-embed")



Found 27599 failed rows to re-embed


In [ ]:
failed_indices


array([43400, 43401, 43402, ..., 70996, 70997, 70998])

In [ ]:
CHECKPOINT_INTERVAL = 700

for count, i in enumerate(failed_indices):
    row = sabdab_pretrain.iloc[i]
    try:
        mean_v, _ = embed_sequence_esm2(row['ag'])
        sab_ag_embs[i] = mean_v
    except Exception as e:
        print(f'  ✗ row {i}: {e}')

    if (count + 1) % CHECKPOINT_INTERVAL == 0:
        np.save(PARTIAL_CACHE, sab_ag_embs)
        print(f'  {count+1}/{len(failed_indices)} repaired — checkpoint saved')

# Save final result
np.save(SAB_MEAN_CACHE, sab_ag_embs)
print(f'Done. Final shape: {sab_ag_embs.shape}')

  700/27599 repaired — checkpoint saved
  1400/27599 repaired — checkpoint saved
  2100/27599 repaired — checkpoint saved
  2800/27599 repaired — checkpoint saved
  3500/27599 repaired — checkpoint saved
  4200/27599 repaired — checkpoint saved
  4900/27599 repaired — checkpoint saved
  5600/27599 repaired — checkpoint saved
  6300/27599 repaired — checkpoint saved
  7000/27599 repaired — checkpoint saved
  7700/27599 repaired — checkpoint saved
  8400/27599 repaired — checkpoint saved
  9100/27599 repaired — checkpoint saved
  9800/27599 repaired — checkpoint saved
  10500/27599 repaired — checkpoint saved
  11200/27599 repaired — checkpoint saved
  11900/27599 repaired — checkpoint saved
  12600/27599 repaired — checkpoint saved
  13300/27599 repaired — checkpoint saved
  14000/27599 repaired — checkpoint saved
  14700/27599 repaired — checkpoint saved
  15400/27599 repaired — checkpoint saved
  16100/27599 repaired — checkpoint saved
  16800/27599 repaired — checkpoint saved
  17500

In [ ]:
len(sab_ag_embs)

70999

In [ ]:
# ── Checkpoint after Section 1.1d ────────────────────────────────────────────
np.save(SAB_MEAN_CACHE, sab_ag_embs)  # already saved in block 3, but be safe
sabdab_pretrain.to_csv(f'{PROJECT_DIR}/abrank_stage1.csv', index=False)
sabdab_finetune.to_csv(f'{PROJECT_DIR}/abrank_stage2.csv', index=False)
#df.to_csv(f'{PROJECT_DIR}/clinical_mabs_with_antigens.csv', index=False)

print('Checkpoint saved:')
print(f'  sab_ag_embs:      {sab_ag_embs.shape}')
print(f'  sabdab_pretrain:  {len(sabdab_pretrain)} rows')
print(f'  sabdab_finetune:  {len(sabdab_finetune)} rows')
#print(f'  df:               {len(df)} rows')

Checkpoint saved:
  sab_ag_embs:      (70999, 1280)
  sabdab_pretrain:  70999 rows
  sabdab_finetune:  4610 rows


## Preprocessing Complete

All outputs saved to Drive. Proceed to `notebook2_embeddings_oracle.ipynb`.